# 03 — Data Integration

## Olist E-Commerce Business Analytics

### Objective

This notebook integrates the cleaned relational datasets into analysis-ready tables while preserving the correct level of detail for each business question.

The integration process will:

- Load and validate the processed datasets
- Document the grain of each source table
- Examine one-to-one and one-to-many relationships
- Prevent revenue inflation during joins
- Define consistent revenue measures
- Create order-level, customer-level, and seller-level analytical tables
- Validate row counts and financial totals after integration

The source tables will remain available separately because no single merged table can safely support every type of analysis.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

PROCESSED_DATA_DIR = Path("../data/processed").resolve()

print("Processed data directory:")
print(PROCESSED_DATA_DIR)

print("\nDirectory exists:")
print(PROCESSED_DATA_DIR.exists())

Processed data directory:
/Users/saadmaher/Desktop/Data Science/Project portfolio/olist-ecommerce-business-analytics/data/processed

Directory exists:
True


## 1. Load the Processed Data

The cleaned datasets are loaded from `data/processed/`. These files contain the documented transformations and quality indicators created in the previous notebook.

In [3]:
dtype_rules = {
    "customers": {
        "customer_zip_code_prefix": "string"
    },
    "sellers": {
        "seller_zip_code_prefix": "string"
    },
    "geolocation": {
        "geolocation_zip_code_prefix": "string"
    },
    "payments": {
        "payment_installments": "Int64"
    }
}

datasets = {
    dataset_name: pd.read_csv(
        PROCESSED_DATA_DIR / filename,
        dtype=dtype_rules.get(dataset_name)
    )
    for dataset_name, filename in processed_files.items()
}

print(f"Processed datasets loaded: {len(datasets)}")

Processed datasets loaded: 9


### 1.1 Restore Analytical Data Types

CSV files do not preserve Pandas data types. Timestamp columns must therefore be converted back to datetime after loading, while ZIP-code prefixes are explicitly loaded as text to preserve leading zeros.

In [4]:
datetime_columns = {
    "orders": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ],
    "order_items": [
        "shipping_limit_date"
    ],
    "reviews": [
        "review_creation_date",
        "review_answer_timestamp"
    ]
}

datetime_validation_records = []

for dataset_name, columns in datetime_columns.items():
    for column in columns:
        datasets[dataset_name][column] = pd.to_datetime(
            datasets[dataset_name][column],
            errors="coerce"
        )

        datetime_validation_records.append({
            "dataset": dataset_name,
            "column": column,
            "data_type": str(
                datasets[dataset_name][column].dtype
            )
        })

datetime_type_summary = pd.DataFrame(
    datetime_validation_records
)

datetime_type_summary

,dataset,column,data_type
0,orders,order_purchase_timestamp,datetime64[ns]
1,orders,order_approved_at,datetime64[ns]
2,orders,order_delivered_carrier_date,datetime64[ns]
3,orders,order_delivered_customer_date,datetime64[ns]
4,orders,order_estimated_delivery_date,datetime64[ns]
5,order_items,shipping_limit_date,datetime64[ns]
6,reviews,review_creation_date,datetime64[ns]
7,reviews,review_answer_timestamp,datetime64[ns]


In [5]:
zip_type_validation = pd.DataFrame([
    {
        "dataset": "customers",
        "data_type": str(
            datasets["customers"][
                "customer_zip_code_prefix"
            ].dtype
        ),
        "invalid_lengths": (
            datasets["customers"][
                "customer_zip_code_prefix"
            ].str.len() != 5
        ).sum()
    },
    {
        "dataset": "sellers",
        "data_type": str(
            datasets["sellers"][
                "seller_zip_code_prefix"
            ].dtype
        ),
        "invalid_lengths": (
            datasets["sellers"][
                "seller_zip_code_prefix"
            ].str.len() != 5
        ).sum()
    },
    {
        "dataset": "geolocation",
        "data_type": str(
            datasets["geolocation"][
                "geolocation_zip_code_prefix"
            ].dtype
        ),
        "invalid_lengths": (
            datasets["geolocation"][
                "geolocation_zip_code_prefix"
            ].str.len() != 5
        ).sum()
    }
])

zip_type_validation

,dataset,data_type,invalid_lengths
0,customers,string,0
1,sellers,string,0
2,geolocation,string,0


## 2. Define Table Grain

Each relational table represents a different level of detail. Documenting the grain prevents incorrect joins, duplicated transactions, and inflated financial metrics.

In [6]:
table_grain_definitions = {
    "customers": {
        "grain": "One customer record associated with an order",
        "key": "customer_id"
    },
    "orders": {
        "grain": "One row per order",
        "key": "order_id"
    },
    "order_items": {
        "grain": "One row per item position within an order",
        "key": "order_id + order_item_id"
    },
    "payments": {
        "grain": "One row per payment component within an order",
        "key": "order_id + payment_sequential"
    },
    "reviews": {
        "grain": "One row per review record associated with an order",
        "key": "review_id + order_id"
    },
    "products": {
        "grain": "One row per product",
        "key": "product_id"
    },
    "sellers": {
        "grain": "One row per seller",
        "key": "seller_id"
    },
    "geolocation": {
        "grain": "One coordinate record within a ZIP-code prefix",
        "key": "No natural unique key"
    },
    "category_translation": {
        "grain": "One English translation per Portuguese category",
        "key": "product_category_name"
    }
}

grain_records = []

for dataset_name, definition in table_grain_definitions.items():
    grain_records.append({
        "dataset": dataset_name,
        "rows": len(datasets[dataset_name]),
        "grain": definition["grain"],
        "key": definition["key"]
    })

table_grain_summary = pd.DataFrame(grain_records)

table_grain_summary.style.format({
    "rows": "{:,.0f}"
})

,dataset,rows,grain,key
0,customers,"99,441",One customer record associated with an order,customer_id
1,orders,"99,441",One row per order,order_id
2,order_items,"112,650",One row per item position within an order,order_id + order_item_id
3,payments,"103,886",One row per payment component within an order,order_id + payment_sequential
4,reviews,"99,224",One row per review record associated with an order,review_id + order_id
5,products,"32,951",One row per product,product_id
6,sellers,"3,095",One row per seller,seller_id
7,geolocation,"738,332",One coordinate record within a ZIP-code prefix,No natural unique key
8,category_translation,74,One English translation per Portuguese category,product_category_name


***Observation:*** The Olist datasets operate at several levels of detail, including order, order item, payment component, review, product, seller, and geographic coordinate levels.

***Integration rule:*** One-to-many tables must be aggregated to the required analytical grain before they are joined. Directly joining order items, payments, and reviews together could create a many-to-many relationship and inflate revenue, freight, payment, and review metrics.

## 3. Relationship Cardinality

Cardinality describes how many child records can be associated with one parent record. Measuring cardinality determines which tables must be aggregated before integration.

In [7]:
cardinality_definitions = {
    "order_items": datasets["order_items"]["order_id"],
    "payments": datasets["payments"]["order_id"],
    "reviews": datasets["reviews"]["order_id"]
}

cardinality_records = []

for dataset_name, order_id_series in cardinality_definitions.items():
    records_per_order = order_id_series.value_counts()

    cardinality_records.append({
        "dataset": dataset_name,
        "orders_represented": records_per_order.size,
        "orders_with_multiple_records": (
            records_per_order > 1
        ).sum(),
        "average_records_per_order": records_per_order.mean(),
        "maximum_records_per_order": records_per_order.max()
    })

cardinality_summary = pd.DataFrame(
    cardinality_records
)

cardinality_summary.style.format({
    "orders_represented": "{:,.0f}",
    "orders_with_multiple_records": "{:,.0f}",
    "average_records_per_order": "{:.2f}",
    "maximum_records_per_order": "{:,.0f}"
})

,dataset,orders_represented,orders_with_multiple_records,average_records_per_order,maximum_records_per_order
0,order_items,"98,666","9,803",1.14,21
1,payments,"99,440","2,961",1.04,29
2,reviews,"98,673",547,1.01,3


***Observation:*** One-to-many relationships are present in all three order-linked detail tables. Order items have the greatest multi-record frequency, while payments and reviews also contain orders represented by multiple rows.

***Integration decision:*** Order items, payments, and reviews will be aggregated independently to one row per order before being joined. This prevents Cartesian multiplication and protects item revenue, freight, payment, and satisfaction metrics from inflation.

## 4. Financial Measure Definitions

The Olist data contains several monetary fields that answer different business questions. They must not be treated as interchangeable measures.

- **Product value:** Sum of `price` from order items. Represents the listed value of products sold.
- **Freight value:** Sum of `freight_value` from order items. Represents delivery charges associated with items.
- **Item total value:** Product value plus freight value.
- **Payment value:** Sum of `payment_value`. Represents the amount recorded across customer payment components.

Payment value will be used for customer-payment analysis, while product and seller analyses will use item-level product value. Differences between item totals and payment totals will be measured and documented rather than silently reconciled.

In [8]:
order_items = datasets["order_items"]
payments = datasets["payments"]

financial_measure_summary = pd.DataFrame([
    {
        "financial_measure": "Product value",
        "source": "order_items.price",
        "total_value": order_items["price"].sum()
    },
    {
        "financial_measure": "Freight value",
        "source": "order_items.freight_value",
        "total_value": order_items["freight_value"].sum()
    },
    {
        "financial_measure": "Item total value",
        "source": "price + freight_value",
        "total_value": (
            order_items["price"].sum()
            + order_items["freight_value"].sum()
        )
    },
    {
        "financial_measure": "Payment value",
        "source": "payments.payment_value",
        "total_value": payments["payment_value"].sum()
    }
])

financial_measure_summary.style.format({
    "total_value": "R$ {:,.2f}"
})

,financial_measure,source,total_value
0,Product value,order_items.price,"R$ 13,591,643.70"
1,Freight value,order_items.freight_value,"R$ 2,251,909.54"
2,Item total value,price + freight_value,"R$ 15,843,553.24"
3,Payment value,payments.payment_value,"R$ 16,008,872.12"


In [9]:
total_item_value = (
    order_items["price"].sum()
    + order_items["freight_value"].sum()
)

total_payment_value = payments["payment_value"].sum()

overall_value_difference = (
    total_payment_value - total_item_value
)

print(
    "Payment value minus item total value:",
    f"R$ {overall_value_difference:,.2f}"
)

Payment value minus item total value: R$ 165,318.88


***Observation:*** Product value totals R$13.59 million, freight totals R$2.25 million, and their combined item value is R$15.84 million. Recorded payment value is R$16.01 million, exceeding item total value by R$165,318.88, or approximately 1.04%.

***Interpretation:*** The difference may reflect orders without item records, cancellations, payment adjustments, or other marketplace payment mechanics. It will be investigated at order level rather than treated automatically as a data error.

***Metric decision:*** The project will retain separate product, freight, item-total, and payment measures. Business conclusions will always specify which monetary definition is being used.

## 5. Create Order-Level Item Summary

The order-items table contains one row per item position. It must be aggregated to one row per order before integration with orders, payments, or reviews.

In [10]:
order_item_summary = (
    datasets["order_items"]
    .groupby("order_id", as_index=False)
    .agg(
        item_count=("order_item_id", "count"),
        distinct_product_count=("product_id", "nunique"),
        distinct_seller_count=("seller_id", "nunique"),
        product_value=("price", "sum"),
        freight_value=("freight_value", "sum"),
        average_item_price=("price", "mean"),
        maximum_shipping_limit=("shipping_limit_date", "max")
    )
)

order_item_summary["item_total_value"] = (
    order_item_summary["product_value"]
    + order_item_summary["freight_value"]
)

print("Order-item source rows:", f"{len(datasets['order_items']):,}")
print("Orders represented:", f"{len(order_item_summary):,}")
print(
    "Duplicate order IDs:",
    order_item_summary["order_id"].duplicated().sum()
)

order_item_summary.head()

Order-item source rows: 112,650
Orders represented: 98,666
Duplicate order IDs: 0


,order_id,item_count,distinct_product_count,distinct_seller_count,product_value,freight_value,average_item_price,maximum_shipping_limit,item_total_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,1,1,58.90,13.29,58.90,2017-09-19 09:45:35,72.19
1,00018f77f2f0320c557190d7a144bdd3,1,1,1,239.90,19.93,239.90,2017-05-03 11:05:13,259.83
2,000229ec398224ef6ca0657da4fc703e,1,1,1,199.00,17.87,199.00,2018-01-18 14:48:30,216.87
3,00024acbcdf0a6daa1e931b038114c75,1,1,1,12.99,12.79,12.99,2018-08-15 10:10:18,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,1,1,199.90,18.14,199.90,2017-02-13 13:57:51,218.04


In [11]:
item_aggregation_validation = pd.DataFrame([
    {
        "measure": "Product value",
        "source_total": datasets["order_items"]["price"].sum(),
        "aggregated_total": order_item_summary["product_value"].sum()
    },
    {
        "measure": "Freight value",
        "source_total": datasets["order_items"]["freight_value"].sum(),
        "aggregated_total": order_item_summary["freight_value"].sum()
    }
])

item_aggregation_validation["difference"] = (
    item_aggregation_validation["aggregated_total"]
    - item_aggregation_validation["source_total"]
)

item_aggregation_validation.style.format({
    "source_total": "R$ {:,.2f}",
    "aggregated_total": "R$ {:,.2f}",
    "difference": "R$ {:,.2f}"
})

,measure,source_total,aggregated_total,difference
0,Product value,"R$ 13,591,643.70","R$ 13,591,643.70",R$ 0.00
1,Freight value,"R$ 2,251,909.54","R$ 2,251,909.54",R$ 0.00


***Observation:*** The 112,650 order-item records were aggregated into 98,666 unique orders. The resulting table contains no duplicate `order_id` values.

***Validation:*** Aggregated product and freight values reconcile exactly with their source totals, with differences of R$0.00. This confirms that the aggregation changed the level of detail without losing or duplicating financial value.

***Integration decision:*** `order_item_summary` can now be joined safely to the one-row-per-order orders table.

## 6. Create Order-Level Payment Summary

An order can contain multiple payment components, including combinations of cards and vouchers. Payment records will be aggregated to one row per order while preserving total payment value and payment behaviour.

In [12]:
payments = datasets["payments"]

payment_summary = (
    payments
    .groupby("order_id", as_index=False)
    .agg(
        payment_record_count=(
            "payment_sequential",
            "count"
        ),
        payment_method_count=(
            "payment_type",
            "nunique"
        ),
        total_payment_value=(
            "payment_value",
            "sum"
        ),
        maximum_installments=(
            "payment_installments",
            "max"
        ),
        has_zero_payment=(
            "is_zero_payment",
            "max"
        ),
        has_payment_anomaly=(
            "has_payment_anomaly",
            "max"
        )
    )
)

In [13]:
primary_payment_method = (
    payments
    .sort_values(
        [
            "order_id",
            "payment_value",
            "payment_sequential"
        ],
        ascending=[True, False, True]
    )
    .drop_duplicates(
        subset="order_id",
        keep="first"
    )
    [["order_id", "payment_type"]]
    .rename(
        columns={
            "payment_type": "primary_payment_type"
        }
    )
)

payment_summary = payment_summary.merge(
    primary_payment_method,
    on="order_id",
    how="left",
    validate="one_to_one"
)

print("Payment source rows:", f"{len(payments):,}")
print("Orders represented:", f"{len(payment_summary):,}")
print(
    "Duplicate order IDs:",
    payment_summary["order_id"].duplicated().sum()
)

payment_summary.head()

Payment source rows: 103,886
Orders represented: 99,440
Duplicate order IDs: 0


,order_id,payment_record_count,payment_method_count,total_payment_value,maximum_installments,has_zero_payment,has_payment_anomaly,primary_payment_type
0,00010242fe8c5a6d1ba2dd792cb16214,1,1,72.19,2,False,False,credit_card
1,00018f77f2f0320c557190d7a144bdd3,1,1,259.83,3,False,False,credit_card
2,000229ec398224ef6ca0657da4fc703e,1,1,216.87,5,False,False,credit_card
3,00024acbcdf0a6daa1e931b038114c75,1,1,25.78,2,False,False,credit_card
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,1,218.04,3,False,False,credit_card


In [15]:
payment_difference = (
    payment_aggregated_total
    - payment_source_total
)

if np.isclose(payment_difference, 0, atol=0.01):
    payment_difference = 0.0

payment_aggregation_validation = pd.DataFrame([{
    "measure": "Payment value",
    "source_total": payment_source_total,
    "aggregated_total": payment_aggregated_total,
    "difference": payment_difference
}])

payment_aggregation_validation.style.format({
    "source_total": "R$ {:,.2f}",
    "aggregated_total": "R$ {:,.2f}",
    "difference": "R$ {:,.2f}"
})

,measure,source_total,aggregated_total,difference
0,Payment value,"R$ 16,008,872.12","R$ 16,008,872.12",R$ 0.00


***Observation:*** Payment components were aggregated into 99,440 unique order-level records. The primary payment type represents the payment component with the highest value within each order.

***Validation:*** Aggregated payment value reconciles with the source total of R$16,008,872.12.  
The difference is effectively R$0.00 after accounting for normal floating-point precision.

***Integration decision:*** `payment_summary` can now be joined safely to the orders table without multiplying payment values across item or review records.

## 7. Create Order-Level Review Summary

Most orders have one review, but a small number have multiple review records. Reviews will be summarised to one row per order while preserving review frequency, score variation, written-feedback availability, and the latest recorded score.

In [16]:
reviews = datasets["reviews"]

review_summary = (
    reviews
    .groupby("order_id", as_index=False)
    .agg(
        review_record_count=(
            "review_id",
            "count"
        ),
        average_review_score=(
            "review_score",
            "mean"
        ),
        minimum_review_score=(
            "review_score",
            "min"
        ),
        maximum_review_score=(
            "review_score",
            "max"
        ),
        has_written_feedback=(
            "has_written_feedback",
            "max"
        ),
        written_feedback_count=(
            "has_written_feedback",
            "sum"
        )
    )
)

In [17]:
latest_reviews = (
    reviews
    .sort_values(
        [
            "order_id",
            "review_creation_date",
            "review_answer_timestamp"
        ]
    )
    .drop_duplicates(
        subset="order_id",
        keep="last"
    )
    [
        [
            "order_id",
            "review_score",
            "review_creation_date",
            "review_answer_timestamp"
        ]
    ]
    .rename(
        columns={
            "review_score": "latest_review_score",
            "review_creation_date": "latest_review_date",
            "review_answer_timestamp": "latest_review_answer_timestamp"
        }
    )
)

review_summary = review_summary.merge(
    latest_reviews,
    on="order_id",
    how="left",
    validate="one_to_one"
)

print("Review source rows:", f"{len(reviews):,}")
print("Orders represented:", f"{len(review_summary):,}")
print(
    "Duplicate order IDs:",
    review_summary["order_id"].duplicated().sum()
)

review_summary.head()

Review source rows: 99,224
Orders represented: 98,673
Duplicate order IDs: 0


,order_id,review_record_count,average_review_score,minimum_review_score,maximum_review_score,has_written_feedback,written_feedback_count,latest_review_score,latest_review_date,latest_review_answer_timestamp
0,00010242fe8c5a6d1ba2dd792cb16214,1,5.00,5,5,True,1,5,2017-09-21,2017-09-22 10:57:03
1,00018f77f2f0320c557190d7a144bdd3,1,4.00,4,4,False,0,4,2017-05-13,2017-05-15 11:34:13
2,000229ec398224ef6ca0657da4fc703e,1,5.00,5,5,True,1,5,2018-01-23,2018-01-23 16:06:31
3,00024acbcdf0a6daa1e931b038114c75,1,4.00,4,4,False,0,4,2018-08-15,2018-08-15 16:39:01
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,5.00,5,5,True,1,5,2017-03-02,2017-03-03 10:54:59


In [18]:
review_aggregation_validation = pd.DataFrame([
    {
        "validation_check": "Review records represented",
        "result": review_summary[
            "review_record_count"
        ].sum(),
        "expected": len(reviews)
    },
    {
        "validation_check": "Minimum latest review score",
        "result": review_summary[
            "latest_review_score"
        ].min(),
        "expected": 1
    },
    {
        "validation_check": "Maximum latest review score",
        "result": review_summary[
            "latest_review_score"
        ].max(),
        "expected": 5
    },
    {
        "validation_check": "Duplicate order IDs",
        "result": review_summary[
            "order_id"
        ].duplicated().sum(),
        "expected": 0
    }
])

review_aggregation_validation["status"] = np.where(
    review_aggregation_validation["result"]
    == review_aggregation_validation["expected"],
    "Passed",
    "Failed"
)

review_aggregation_validation

,validation_check,result,expected,status
0,Review records represented,99224,99224,Passed
1,Minimum latest review score,1,1,Passed
2,Maximum latest review score,5,5,Passed
3,Duplicate order IDs,0,0,Passed


***Observation:*** The 99,224 review records were aggregated into 98,673 unique order-level summaries without losing any review records. Latest review scores remain within the valid 1–5 range.

***Integration decision:*** Both average and latest review scores were retained because some orders have multiple review records. The latest score will represent the final recorded customer evaluation, while the average, minimum, and maximum scores preserve score variation.

***Validation:*** The sum of `review_record_count` matches the original review-table row count, all score-range checks passed, and no duplicate order IDs remain.

## 8. Create the Order-Level Analytical Table

The cleaned orders table provides the base grain of one row per order. Customer attributes and the three aggregated summaries will be joined using validated relationships.

In [19]:
orders = datasets["orders"]

customer_columns = [
    "customer_id",
    "customer_unique_id",
    "customer_zip_code_prefix",
    "customer_city",
    "customer_state"
]

order_analytics = (
    orders
    .merge(
        datasets["customers"][customer_columns],
        on="customer_id",
        how="left",
        validate="many_to_one"
    )
    .merge(
        order_item_summary,
        on="order_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        payment_summary,
        on="order_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        review_summary,
        on="order_id",
        how="left",
        validate="one_to_one"
    )
)

print("Source order rows:", f"{len(orders):,}")
print("Integrated order rows:", f"{len(order_analytics):,}")
print(
    "Duplicate order IDs:",
    order_analytics["order_id"].duplicated().sum()
)
print(
    "Missing customer identities:",
    order_analytics["customer_unique_id"].isna().sum()
)
print("Integrated columns:", order_analytics.shape[1])

Source order rows: 99,441
Integrated order rows: 99,441
Duplicate order IDs: 0
Missing customer identities: 0
Integrated columns: 45


In [20]:
order_analytics.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,carrier_before_purchase,delivery_before_purchase,delivery_before_carrier,has_invalid_timestamp_sequence,has_complete_delivery_timestamps,is_valid_for_delivery_analysis,has_order_items,has_payment,has_review,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,item_count,distinct_product_count,distinct_seller_count,product_value,freight_value,average_item_price,maximum_shipping_limit,item_total_value,payment_record_count,payment_method_count,total_payment_value,maximum_installments,has_zero_payment,has_payment_anomaly,primary_payment_type,review_record_count,average_review_score,minimum_review_score,maximum_review_score,has_written_feedback,written_feedback_count,latest_review_score,latest_review_date,latest_review_answer_timestamp
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,False,False,False,False,True,True,True,True,True,7c396fd4830fd04220f754e42b4e5bff,03149,sao paulo,SP,1.00,1.00,1.00,29.99,8.72,29.99,2017-10-06 11:07:15,38.71,3.00,2.00,38.71,1,False,False,voucher,1.00,4.00,4.00,4.00,True,1.00,4.00,2017-10-11,2017-10-12 03:43:48
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,False,False,False,False,True,True,True,True,True,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,1.00,1.00,1.00,118.70,22.76,118.70,2018-07-30 03:24:27,141.46,1.00,1.00,141.46,1,False,False,boleto,1.00,4.00,4.00,4.00,True,1.00,4.00,2018-08-08,2018-08-08 18:37:50
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,False,False,False,False,True,True,True,True,True,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,1.00,1.00,1.00,159.90,19.22,159.90,2018-08-13 08:55:23,179.12,1.00,1.00,179.12,3,False,False,credit_card,1.00,5.00,5.00,5.00,False,0.00,5.00,2018-08-18,2018-08-22 19:07:58
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,False,False,False,False,True,True,True,True,True,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,1.00,1.00,1.00,45.00,27.20,45.00,2017-11-23 19:45:59,72.20,1.00,1.00,72.20,1,False,False,credit_card,1.00,5.00,5.00,5.00,True,1.00,5.00,2017-12-03,2017-12-05 19:21:58
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,False,False,False,False,True,True,True,True,True,72632f0f9dd73dfee390c9b22eb56dd6,09195,santo andre,SP,1.00,1.00,1.00,19.90,8.72,19.90,2018-02-19 20:31:37,28.62,1.00,1.00,28.62,1,False,False,credit_card,1.00,5.00,5.00,5.00,False,0.00,5.00,2018-02-17,2018-02-18 13:02:51


***Observation:*** The central order-level table contains all 99,441 source orders and 45 analytical columns. No duplicate order IDs or missing customer identities were introduced during integration.

***Validation:*** The one-row-per-order grain was preserved through all four joins. Orders without items, payments, or reviews remain in the table with missing measures rather than being removed.

### 8.1 Standardise Missing Summary Values

Missing counts caused by absent child records will be converted to zero. Missing financial values, ratings, installment values, and timestamps will remain null to distinguish unavailable information from genuine zero values.

In [22]:
count_columns = [
    "item_count",
    "distinct_product_count",
    "distinct_seller_count",
    "payment_record_count",
    "payment_method_count",
    "review_record_count",
    "written_feedback_count"
]

boolean_columns = [
    "has_zero_payment",
    "has_payment_anomaly",
    "has_written_feedback"
]

for column in count_columns:
    order_analytics[column] = (
        order_analytics[column]
        .fillna(0)
        .astype("Int64")
    )

for column in boolean_columns:
    order_analytics[column] = (
        order_analytics[column]
        .astype("boolean")
        .fillna(False)
    )

missing_summary_validation = pd.DataFrame({
    "column": count_columns + boolean_columns,
    "missing_values_after": [
        order_analytics[column].isna().sum()
        for column in count_columns + boolean_columns
    ]
})

missing_summary_validation

,column,missing_values_after
0,item_count,0
1,distinct_product_count,0
2,distinct_seller_count,0
3,payment_record_count,0
4,payment_method_count,0
5,review_record_count,0
6,written_feedback_count,0
7,has_zero_payment,0
8,has_payment_anomaly,0
9,has_written_feedback,0


***Integration decision:*** Missing item, payment, and review counts were converted to zero because the absence of a related record means no records were observed. Related Boolean indicators were converted to `False`.

***Validation:*** All summary count and Boolean columns are complete after standardisation. Missing financial values, review scores, installments, and timestamps remain null because their values are unknown rather than zero.

### 8.2 Financial Reconciliation After Integration

Financial totals in the integrated order table are reconciled against the independently aggregated item and payment summaries. Matching totals confirm that the joins did not duplicate or remove monetary values.

In [23]:
financial_reconciliation = pd.DataFrame([
    {
        "measure": "Product value",
        "source_total": order_item_summary[
            "product_value"
        ].sum(),
        "integrated_total": order_analytics[
            "product_value"
        ].sum()
    },
    {
        "measure": "Freight value",
        "source_total": order_item_summary[
            "freight_value"
        ].sum(),
        "integrated_total": order_analytics[
            "freight_value"
        ].sum()
    },
    {
        "measure": "Item total value",
        "source_total": order_item_summary[
            "item_total_value"
        ].sum(),
        "integrated_total": order_analytics[
            "item_total_value"
        ].sum()
    },
    {
        "measure": "Payment value",
        "source_total": payment_summary[
            "total_payment_value"
        ].sum(),
        "integrated_total": order_analytics[
            "total_payment_value"
        ].sum()
    }
])

financial_reconciliation["difference"] = (
    financial_reconciliation["integrated_total"]
    - financial_reconciliation["source_total"]
)

financial_reconciliation["difference"] = (
    financial_reconciliation["difference"]
    .where(
        ~np.isclose(
            financial_reconciliation["difference"],
            0,
            atol=0.01
        ),
        0.0
    )
)

financial_reconciliation["status"] = np.where(
    np.isclose(
        financial_reconciliation["source_total"],
        financial_reconciliation["integrated_total"],
        atol=0.01
    ),
    "Passed",
    "Failed"
)

financial_reconciliation.style.format({
    "source_total": "R$ {:,.2f}",
    "integrated_total": "R$ {:,.2f}",
    "difference": "R$ {:,.2f}"
})

,measure,source_total,integrated_total,difference,status
0,Product value,"R$ 13,591,643.70","R$ 13,591,643.70",R$ 0.00,Passed
1,Freight value,"R$ 2,251,909.54","R$ 2,251,909.54",R$ 0.00,Passed
2,Item total value,"R$ 15,843,553.24","R$ 15,843,553.24",R$ 0.00,Passed
3,Payment value,"R$ 16,008,872.12","R$ 16,008,872.12",R$ 0.00,Passed


***Validation:*** Product, freight, item-total, and payment values reconcile exactly between the aggregated source tables and the integrated order table. This confirms that the joins preserved all monetary values without creating duplication.

***Integration result:*** The order-level table is financially reliable and can support subsequent KPI calculations using explicitly defined monetary measures.

## 9. Create Time and Delivery Features

Calendar and delivery features are derived from the validated order timestamps. Delivery durations are calculated only for orders marked as valid for delivery analysis.

In [24]:
order_analytics["purchase_date"] = (
    order_analytics["order_purchase_timestamp"].dt.date
)

order_analytics["purchase_year"] = (
    order_analytics["order_purchase_timestamp"].dt.year
    .astype("Int64")
)

order_analytics["purchase_month"] = (
    order_analytics["order_purchase_timestamp"].dt.month
    .astype("Int64")
)

order_analytics["purchase_year_month"] = (
    order_analytics["order_purchase_timestamp"]
    .dt.to_period("M")
    .astype("string")
)

order_analytics["purchase_quarter"] = (
    order_analytics["order_purchase_timestamp"]
    .dt.to_period("Q")
    .astype("string")
)

valid_delivery_mask = (
    order_analytics["is_valid_for_delivery_analysis"]
)

order_analytics["delivery_days"] = np.nan
order_analytics.loc[
    valid_delivery_mask,
    "delivery_days"
] = (
    (
        order_analytics.loc[
            valid_delivery_mask,
            "order_delivered_customer_date"
        ]
        - order_analytics.loc[
            valid_delivery_mask,
            "order_purchase_timestamp"
        ]
    )
    .dt.total_seconds()
    / 86400
)

order_analytics["carrier_to_customer_days"] = np.nan
order_analytics.loc[
    valid_delivery_mask,
    "carrier_to_customer_days"
] = (
    (
        order_analytics.loc[
            valid_delivery_mask,
            "order_delivered_customer_date"
        ]
        - order_analytics.loc[
            valid_delivery_mask,
            "order_delivered_carrier_date"
        ]
    )
    .dt.total_seconds()
    / 86400
)

In [25]:
actual_delivery_date = (
    order_analytics["order_delivered_customer_date"]
    .dt.normalize()
)

estimated_delivery_date = (
    order_analytics["order_estimated_delivery_date"]
    .dt.normalize()
)

order_analytics["delivery_delay_days"] = np.nan

order_analytics.loc[
    valid_delivery_mask,
    "delivery_delay_days"
] = (
    actual_delivery_date[valid_delivery_mask]
    - estimated_delivery_date[valid_delivery_mask]
).dt.days

order_analytics["days_late"] = (
    order_analytics["delivery_delay_days"]
    .clip(lower=0)
)

order_analytics["is_late_delivery"] = pd.Series(
    pd.NA,
    index=order_analytics.index,
    dtype="boolean"
)

order_analytics.loc[
    valid_delivery_mask,
    "is_late_delivery"
] = (
    order_analytics.loc[
        valid_delivery_mask,
        "delivery_delay_days"
    ] > 0
)

order_analytics["is_on_time_delivery"] = pd.Series(
    pd.NA,
    index=order_analytics.index,
    dtype="boolean"
)

order_analytics.loc[
    valid_delivery_mask,
    "is_on_time_delivery"
] = (
    order_analytics.loc[
        valid_delivery_mask,
        "delivery_delay_days"
    ] <= 0
)

In [26]:
delivery_feature_summary = pd.DataFrame([
    {
        "metric": "Valid delivery orders",
        "value": valid_delivery_mask.sum()
    },
    {
        "metric": "Delivery durations calculated",
        "value": order_analytics["delivery_days"].notna().sum()
    },
    {
        "metric": "Late deliveries",
        "value": (
            order_analytics["is_late_delivery"] == True
        ).sum()
    },
    {
        "metric": "On-time deliveries",
        "value": (
            order_analytics["is_on_time_delivery"] == True
        ).sum()
    }
])

delivery_feature_summary.style.format({
    "value": "{:,.0f}"
})

,metric,value
0,Valid delivery orders,"96,281"
1,Delivery durations calculated,"96,281"
2,Late deliveries,"6,531"
3,On-time deliveries,"89,750"


***Observation:*** Of the 96,281 orders eligible for delivery analysis, 89,750 were delivered on or before the estimated date and 6,531 were delivered late. This corresponds to an on-time delivery rate of approximately 93.22%.

***Methodological clarification:*** The earlier data-quality check identified 7,827 deliveries whose full timestamp exceeded the estimated timestamp. The integrated metric uses calendar dates and excludes invalid timestamp sequences, producing the more appropriate count of 6,531 late deliveries. Orders delivered later during their estimated calendar date are classified as on time.

***Validation:*** Late and on-time delivery counts sum exactly to the 96,281 valid delivery orders.

## 10. Create the Customer-Level Analytical Table

The customer-level table contains one row per unique customer. It summarises observed purchasing activity, payment value, review scores, delivery performance, and the customer’s latest recorded location.

This table provides the foundation for later RFM, customer-value, cohort, retention, and inactivity analyses. It does not yet assign churn or CLV labels.

In [27]:
order_analytics["is_delivered_order"] = (
    order_analytics["order_status"] == "delivered"
)

order_analytics["has_valid_review_score"] = (
    order_analytics["latest_review_score"].notna()
)

order_analytics["has_valid_delivery_metric"] = (
    order_analytics["is_late_delivery"].notna()
)

order_analytics["late_delivery_flag_numeric"] = (
    order_analytics["is_late_delivery"]
    .fillna(False)
    .astype(int)
)

In [28]:
customer_analytics = (
    order_analytics
    .groupby("customer_unique_id", as_index=False)
    .agg(
        order_count=(
            "order_id",
            "nunique"
        ),
        first_purchase_timestamp=(
            "order_purchase_timestamp",
            "min"
        ),
        last_purchase_timestamp=(
            "order_purchase_timestamp",
            "max"
        ),
        delivered_order_count=(
            "is_delivered_order",
            "sum"
        ),
        observed_product_value=(
            "product_value",
            lambda series: series.sum(min_count=1)
        ),
        observed_freight_value=(
            "freight_value",
            lambda series: series.sum(min_count=1)
        ),
        observed_payment_value=(
            "total_payment_value",
            lambda series: series.sum(min_count=1)
        ),
        average_order_payment_value=(
            "total_payment_value",
            "mean"
        ),
        reviewed_order_count=(
            "has_valid_review_score",
            "sum"
        ),
        average_review_score=(
            "latest_review_score",
            "mean"
        ),
        valid_delivery_order_count=(
            "has_valid_delivery_metric",
            "sum"
        ),
        late_delivery_count=(
            "late_delivery_flag_numeric",
            "sum"
        ),
        average_delivery_days=(
            "delivery_days",
            "mean"
        )
    )
)

In [29]:
customer_analytics["is_repeat_customer"] = (
    customer_analytics["order_count"] > 1
)

customer_analytics["observed_customer_tenure_days"] = (
    customer_analytics["last_purchase_timestamp"]
    - customer_analytics["first_purchase_timestamp"]
).dt.days

customer_analytics["late_delivery_rate"] = (
    customer_analytics["late_delivery_count"]
    .div(
        customer_analytics[
            "valid_delivery_order_count"
        ].replace(0, np.nan)
    )
)

In [30]:
latest_customer_location = (
    order_analytics
    .sort_values(
        [
            "customer_unique_id",
            "order_purchase_timestamp"
        ]
    )
    .drop_duplicates(
        subset="customer_unique_id",
        keep="last"
    )
    [
        [
            "customer_unique_id",
            "customer_zip_code_prefix",
            "customer_city",
            "customer_state"
        ]
    ]
    .rename(
        columns={
            "customer_zip_code_prefix":
                "latest_customer_zip_code_prefix",
            "customer_city":
                "latest_customer_city",
            "customer_state":
                "latest_customer_state"
        }
    )
)

customer_analytics = customer_analytics.merge(
    latest_customer_location,
    on="customer_unique_id",
    how="left",
    validate="one_to_one"
)

In [31]:
customer_validation = pd.DataFrame([
    {
        "validation_check": "Unique customers",
        "result": len(customer_analytics),
        "expected": datasets["customers"][
            "customer_unique_id"
        ].nunique()
    },
    {
        "validation_check": "Duplicate customer IDs",
        "result": customer_analytics[
            "customer_unique_id"
        ].duplicated().sum(),
        "expected": 0
    },
    {
        "validation_check": "Orders represented",
        "result": customer_analytics[
            "order_count"
        ].sum(),
        "expected": len(order_analytics)
    },
    {
        "validation_check": "Payment value represented",
        "result": round(
            customer_analytics[
                "observed_payment_value"
            ].sum(),
            2
        ),
        "expected": round(
            order_analytics[
                "total_payment_value"
            ].sum(),
            2
        )
    }
])

customer_validation["status"] = np.where(
    np.isclose(
        customer_validation["result"],
        customer_validation["expected"],
        atol=0.01
    ),
    "Passed",
    "Failed"
)

customer_validation

,validation_check,result,expected,status
0,Unique customers,"96,096.00","96,096.00",Passed
1,Duplicate customer IDs,0.00,0.00,Passed
2,Orders represented,"99,441.00","99,441.00",Passed
3,Payment value represented,"16,008,872.12","16,008,872.12",Passed


***Observation:*** The customer-level table contains 96,096 unique customers and no duplicate `customer_unique_id` values. All 99,441 orders and the complete R$16,008,872.12 payment value are represented.

***Metric clarification:*** `observed_payment_value` represents historical payment value recorded within the dataset period. It is not predictive customer lifetime value. Similarly, `is_repeat_customer` reflects observed repeat purchasing and does not establish churn.

***Validation:*** Customer aggregation changed the analytical grain from one row per order to one row per unique customer without losing order counts or payment value.

## 11. Create the Seller-Order Bridge

The seller-order bridge contains one row for each seller participating in an order. This prevents sellers with multiple item rows in the same order from receiving duplicated review and delivery observations.

In [32]:
seller_order_bridge = (
    datasets["order_items"]
    .groupby(
        ["seller_id", "order_id"],
        as_index=False
    )
    .agg(
        seller_order_item_count=(
            "order_item_id",
            "count"
        ),
        seller_order_product_count=(
            "product_id",
            "nunique"
        ),
        seller_order_product_value=(
            "price",
            "sum"
        ),
        seller_order_freight_value=(
            "freight_value",
            "sum"
        )
    )
)

seller_order_bridge["seller_order_total_value"] = (
    seller_order_bridge["seller_order_product_value"]
    + seller_order_bridge["seller_order_freight_value"]
)

In [33]:
seller_order_bridge = seller_order_bridge.merge(
    order_analytics[
        [
            "order_id",
            "order_status",
            "order_purchase_timestamp",
            "customer_unique_id",
            "customer_state",
            "latest_review_score",
            "delivery_days",
            "delivery_delay_days",
            "is_late_delivery",
            "is_valid_for_delivery_analysis"
        ]
    ],
    on="order_id",
    how="left",
    validate="many_to_one"
)

In [34]:
seller_order_bridge = seller_order_bridge.merge(
    datasets["sellers"],
    on="seller_id",
    how="left",
    validate="many_to_one"
)

In [35]:
seller_order_validation = pd.DataFrame([
    {
        "validation_check": "Duplicate seller-order combinations",
        "result": seller_order_bridge.duplicated(
            subset=["seller_id", "order_id"]
        ).sum(),
        "expected": 0
    },
    {
        "validation_check": "Item records represented",
        "result": seller_order_bridge[
            "seller_order_item_count"
        ].sum(),
        "expected": len(datasets["order_items"])
    },
    {
        "validation_check": "Product value represented",
        "result": round(
            seller_order_bridge[
                "seller_order_product_value"
            ].sum(),
            2
        ),
        "expected": round(
            datasets["order_items"]["price"].sum(),
            2
        )
    },
    {
        "validation_check": "Freight value represented",
        "result": round(
            seller_order_bridge[
                "seller_order_freight_value"
            ].sum(),
            2
        ),
        "expected": round(
            datasets["order_items"][
                "freight_value"
            ].sum(),
            2
        )
    },
    {
        "validation_check": "Missing seller locations",
        "result": seller_order_bridge[
            "seller_state"
        ].isna().sum(),
        "expected": 0
    }
])

seller_order_validation["status"] = np.where(
    np.isclose(
        seller_order_validation["result"],
        seller_order_validation["expected"],
        atol=0.01
    ),
    "Passed",
    "Failed"
)

print(
    "Seller-order combinations:",
    f"{len(seller_order_bridge):,}"
)

seller_order_validation

Seller-order combinations: 100,010


,validation_check,result,expected,status
0,Duplicate seller-order combinations,0.00,0.00,Passed
1,Item records represented,"112,650.00","112,650.00",Passed
2,Product value represented,"13,591,643.70","13,591,643.70",Passed
3,Freight value represented,"2,251,909.54","2,251,909.54",Passed
4,Missing seller locations,0.00,0.00,Passed


***Observation:*** The seller-order bridge contains unique seller-order combinations and represents all 112,650 item records. Product and freight values reconcile exactly with the source data, and no seller locations are missing.

***Interpretation:*** Review and delivery measures are now attached once per seller-order combination rather than once per item, preventing multi-item orders from overweighting seller performance.

### 11.1 Aggregate Seller Performance

The bridge table is aggregated to one row per seller, summarising marketplace activity, observed sales value, customer reach, satisfaction, and delivery performance.

In [36]:
seller_order_bridge["is_delivered_order"] = (
    seller_order_bridge["order_status"] == "delivered"
)

seller_order_bridge["has_valid_review_score"] = (
    seller_order_bridge["latest_review_score"].notna()
)

seller_order_bridge["has_valid_delivery_metric"] = (
    seller_order_bridge["is_late_delivery"].notna()
)

seller_order_bridge["late_delivery_flag_numeric"] = (
    seller_order_bridge["is_late_delivery"]
    .fillna(False)
    .astype(int)
)

In [37]:
seller_analytics = (
    seller_order_bridge
    .groupby("seller_id", as_index=False)
    .agg(
        order_count=(
            "order_id",
            "nunique"
        ),
        item_count=(
            "seller_order_item_count",
            "sum"
        ),
        distinct_customer_count=(
            "customer_unique_id",
            "nunique"
        ),
        first_order_timestamp=(
            "order_purchase_timestamp",
            "min"
        ),
        last_order_timestamp=(
            "order_purchase_timestamp",
            "max"
        ),
        delivered_order_count=(
            "is_delivered_order",
            "sum"
        ),
        observed_product_value=(
            "seller_order_product_value",
            "sum"
        ),
        observed_freight_value=(
            "seller_order_freight_value",
            "sum"
        ),
        average_seller_order_value=(
            "seller_order_product_value",
            "mean"
        ),
        reviewed_order_count=(
            "has_valid_review_score",
            "sum"
        ),
        average_review_score=(
            "latest_review_score",
            "mean"
        ),
        valid_delivery_order_count=(
            "has_valid_delivery_metric",
            "sum"
        ),
        late_delivery_count=(
            "late_delivery_flag_numeric",
            "sum"
        ),
        average_delivery_days=(
            "delivery_days",
            "mean"
        ),
        average_delivery_delay_days=(
            "delivery_delay_days",
            "mean"
        )
    )
)

In [38]:
seller_analytics = seller_analytics.merge(
    datasets["sellers"],
    on="seller_id",
    how="left",
    validate="one_to_one"
)

seller_analytics["late_delivery_rate"] = (
    seller_analytics["late_delivery_count"]
    .div(
        seller_analytics[
            "valid_delivery_order_count"
        ].replace(0, np.nan)
    )
)

seller_analytics["average_review_score"] = (
    seller_analytics["average_review_score"]
    .round(2)
)

In [39]:
seller_validation = pd.DataFrame([
    {
        "validation_check": "Unique sellers",
        "result": len(seller_analytics),
        "expected": datasets["sellers"][
            "seller_id"
        ].nunique()
    },
    {
        "validation_check": "Duplicate seller IDs",
        "result": seller_analytics[
            "seller_id"
        ].duplicated().sum(),
        "expected": 0
    },
    {
        "validation_check": "Item records represented",
        "result": seller_analytics[
            "item_count"
        ].sum(),
        "expected": len(datasets["order_items"])
    },
    {
        "validation_check": "Product value represented",
        "result": round(
            seller_analytics[
                "observed_product_value"
            ].sum(),
            2
        ),
        "expected": round(
            datasets["order_items"]["price"].sum(),
            2
        )
    }
])

seller_validation["status"] = np.where(
    np.isclose(
        seller_validation["result"],
        seller_validation["expected"],
        atol=0.01
    ),
    "Passed",
    "Failed"
)

seller_validation

,validation_check,result,expected,status
0,Unique sellers,"3,095.00","3,095.00",Passed
1,Duplicate seller IDs,0.00,0.00,Passed
2,Item records represented,"112,650.00","112,650.00",Passed
3,Product value represented,"13,591,643.70","13,591,643.70",Passed


***Observation:*** The seller-level table contains all 3,095 sellers with no duplicate seller IDs. All 112,650 item records and the complete R$13,591,643.70 product value are represented.

***Metric limitation:*** Review and delivery measures represent the overall experience of orders involving each seller. For orders containing multiple sellers, the dataset cannot prove which individual seller caused a delay or review outcome.

***Validation:*** Seller aggregation preserved item counts and product value without duplication or loss.

## 12. Final Integration Validation

The final analytical tables are validated for grain, uniqueness, source-record coverage, and financial reconciliation before export.

In [40]:
final_integration_validation = pd.DataFrame([
    {
        "validation_check": "Order-table rows",
        "result": len(order_analytics),
        "expected": len(datasets["orders"])
    },
    {
        "validation_check": "Duplicate order IDs",
        "result": order_analytics[
            "order_id"
        ].duplicated().sum(),
        "expected": 0
    },
    {
        "validation_check": "Customer-table rows",
        "result": len(customer_analytics),
        "expected": datasets["customers"][
            "customer_unique_id"
        ].nunique()
    },
    {
        "validation_check": "Duplicate unique-customer IDs",
        "result": customer_analytics[
            "customer_unique_id"
        ].duplicated().sum(),
        "expected": 0
    },
    {
        "validation_check": "Seller-table rows",
        "result": len(seller_analytics),
        "expected": datasets["sellers"][
            "seller_id"
        ].nunique()
    },
    {
        "validation_check": "Duplicate seller IDs",
        "result": seller_analytics[
            "seller_id"
        ].duplicated().sum(),
        "expected": 0
    },
    {
        "validation_check": "Duplicate seller-order combinations",
        "result": seller_order_bridge.duplicated(
            subset=["seller_id", "order_id"]
        ).sum(),
        "expected": 0
    },
    {
        "validation_check": "Orders represented by customers",
        "result": customer_analytics[
            "order_count"
        ].sum(),
        "expected": len(order_analytics)
    },
    {
        "validation_check": "Items represented by sellers",
        "result": seller_analytics[
            "item_count"
        ].sum(),
        "expected": len(datasets["order_items"])
    },
    {
        "validation_check": "Payment value reconciled",
        "result": round(
            customer_analytics[
                "observed_payment_value"
            ].sum(),
            2
        ),
        "expected": round(
            order_analytics[
                "total_payment_value"
            ].sum(),
            2
        )
    },
    {
        "validation_check": "Product value reconciled",
        "result": round(
            seller_analytics[
                "observed_product_value"
            ].sum(),
            2
        ),
        "expected": round(
            order_analytics[
                "product_value"
            ].sum(),
            2
        )
    }
])

final_integration_validation["status"] = np.where(
    np.isclose(
        final_integration_validation["result"],
        final_integration_validation["expected"],
        atol=0.01
    ),
    "Passed",
    "Failed"
)

final_integration_validation

,validation_check,result,expected,status
0,Order-table rows,"99,441.00","99,441.00",Passed
1,Duplicate order IDs,0.00,0.00,Passed
2,Customer-table rows,"96,096.00","96,096.00",Passed
3,Duplicate unique-customer IDs,0.00,0.00,Passed
4,Seller-table rows,"3,095.00","3,095.00",Passed
5,Duplicate seller IDs,0.00,0.00,Passed
6,Duplicate seller-order combinations,0.00,0.00,Passed
7,Orders represented by customers,"99,441.00","99,441.00",Passed
8,Items represented by sellers,"112,650.00","112,650.00",Passed
9,Payment value reconciled,"16,008,872.12","16,008,872.12",Passed


***Validation objective:*** Every analytical table must retain its intended grain, contain unique keys, represent all relevant source records, and reconcile its financial totals with the order-level table.

***Export rule:*** The analytical tables will only be exported if every final integration validation displays `Passed`.

## 13. Export Analysis-Ready Tables

The validated order-, customer-, seller-order-, and seller-level tables are exported to `data/processed/`. These outputs will support the remaining analytical notebooks, SQL analysis, and Power BI dashboard.

In [41]:
analytical_outputs = {
    "order_analytics.csv": order_analytics,
    "customer_analytics.csv": customer_analytics,
    "seller_order_bridge.csv": seller_order_bridge,
    "seller_analytics.csv": seller_analytics
}

export_records = []

for filename, dataframe in analytical_outputs.items():
    output_path = PROCESSED_DATA_DIR / filename

    dataframe.to_csv(
        output_path,
        index=False,
        date_format="%Y-%m-%d %H:%M:%S"
    )

    export_records.append({
        "output_file": filename,
        "rows": len(dataframe),
        "columns": dataframe.shape[1],
        "file_created": output_path.exists()
    })

integration_validation_path = (
    PROCESSED_DATA_DIR
    / "integration_validation_report.csv"
)

final_integration_validation.to_csv(
    integration_validation_path,
    index=False
)

export_summary = pd.DataFrame(export_records)

export_summary

,output_file,rows,columns,file_created
0,order_analytics.csv,99441,60,True
1,customer_analytics.csv,96096,20,True
2,seller_order_bridge.csv,100010,23,True
3,seller_analytics.csv,3095,20,True


In [42]:
print("Analytical tables exported:", len(export_summary))
print("All analytical files created:", export_summary["file_created"].all())
print(
    "Validation report created:",
    integration_validation_path.exists()
)

Analytical tables exported: 4
All analytical files created: True
Validation report created: True


## Conclusion

***Overall result:*** The cleaned relational data was integrated into validated order-, customer-, seller-order-, and seller-level analytical tables. Each output preserves a clearly defined grain and can support its intended business analysis.

***Financial integrity:*** Product, freight, item-total, and payment values reconcile exactly with their source tables. One-to-many item, payment, and review relationships were aggregated before integration, preventing duplicated financial metrics.

***Analytical boundaries:*** Customer payment value represents observed historical value rather than predictive CLV. Seller review and delivery measures represent order-level experience and do not prove that an individual seller caused a specific outcome.

***Outputs:*** Four analysis-ready tables and one integration-validation report were exported to `data/processed/`.

***Next step:*** `04_exploratory_data_analysis.ipynb` will examine overall business health, order trends, product performance, payment behaviour, review scores, and delivery outcomes.